In [1]:
import os
import PyPDF2
from PyPDF2 import PdfReader
import numpy as np
import pandas as pd
from tabula.io import read_pdf
from datetime import datetime
import re

In [2]:
file_name = r"C:\Users\admin\Documents\22.05.2023 £1,542.83 Ecobat Battery UK Ltd.pdf"
r"C:\Users\admin\Documents\22.05.2023 £1,542.83 Ecobat Battery UK Ltd.pdf"

'C:\\Users\\admin\\Documents\\22.05.2023 £1,542.83 Ecobat Battery UK Ltd.pdf'

In [3]:
invoice_type = "Products"
# inputFolder = os.path.abspath('..\\Forge')

input_file = fr"C:\Users\admin\Downloads\11.10.2023 £1,640.06 Ecobat Battery UK Ltd.pdf"

In [4]:
table1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(260,76.5,285,562),
                  columns=[117,197,257,300,360,400,450,495,517.8,562],
                  pandas_options={'header': None},
                  encoding="windows-1254")

heading = table1[0]
display(heading)

name = "Ecobat Battery"
docnum = heading[7][0]
print(docnum)

date = heading[9][0]
date = str(datetime.strptime(date, "%d/%m/%Y"))
print(date)

ordernum = heading[1][0]
print(ordernum)

transfernum = None
print(transfernum)

,0,1,2,3,4,5,6,7,8,9
0,Order No.,PO27294,Sales Order No.,99682612,Delivery Note.,856310,Invoice No.,23245004,Date,11/10/2023


23245004
2023-10-11 00:00:00
PO27294
None


In [5]:
with open(input_file,'rb') as pdf_file:
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    num_pages = len(pdf_reader.pages)

print(num_pages)

2


In [6]:
#create loop to go through the pages
all_content = []
for page in range(1, num_pages + 1):
    table2 = read_pdf(input_file,
                    pages= page,
                    silent=True,
                    guess=False,
                    area=(303,9,540,494),
                    columns=[76,140,318,345,383,411,450,494],
                    pandas_options={'header': None},
                    encoding="windows-1254")
    contenti = table2[0]
    all_content.append(contenti)

content = pd.concat(all_content).reset_index(drop=True)
display(content)

,0,1,2,3,4,5,6,7
0,LSLA4-6,NaN,LUCAS 6V 4AH AGM STANDBY BATTERY,40.0,9.89,62.0,3.76,150.40
1,LSLA7-12,NaN,LUCAS 12V 7AH AGM STANDBY,14.0,8.71,NaN,8.71,121.94
2,NaN,NaN,BATTERY,NaN,NaN,NaN,NaN,NaN
3,LSLA4.5-12,NaN,LUCAS 12V 4.5AH AGM STANDBY,16.0,19.57,62.0,7.44,119.04
4,NaN,NaN,BATTERY,NaN,NaN,NaN,NaN,NaN
5,LSLA5-12,NaN,LUCAS 12V 5AH AGM STANDBY,10.0,24.18,62.0,9.19,91.90
6,NaN,NaN,BATTERY,NaN,NaN,NaN,NaN,NaN
7,EK131,NaN,EXIDE Auxiliary Battery - EK131,2.0,69.54,54.0,31.99,63.98
8,LSLC22-12G,NaN,LUCAS 12V 22AH AGM CYCLIC GOLF,2.0,76.60,62.0,29.11,58.22
9,NaN,NaN,BATTERY,NaN,NaN,NaN,NaN,NaN


In [7]:

content = content[content[0].notnull()].iloc[:, 0:].reset_index(drop=True)  # Remove NaN
content = content.dropna(subset=[3])  # Remove rows with NaN in column 3
content[3] = pd.to_numeric(content[3], errors='coerce')  # Remove anything that not number in column 1

content

,0,1,2,3,4,5,6,7
0,LSLA4-6,NaN,LUCAS 6V 4AH AGM STANDBY BATTERY,40.0,9.89,62.0,3.76,150.40
1,LSLA7-12,NaN,LUCAS 12V 7AH AGM STANDBY,14.0,8.71,NaN,8.71,121.94
2,LSLA4.5-12,NaN,LUCAS 12V 4.5AH AGM STANDBY,16.0,19.57,62.0,7.44,119.04
3,LSLA5-12,NaN,LUCAS 12V 5AH AGM STANDBY,10.0,24.18,62.0,9.19,91.90
4,EK131,NaN,EXIDE Auxiliary Battery - EK131,2.0,69.54,54.0,31.99,63.98
5,LSLC22-12G,NaN,LUCAS 12V 22AH AGM CYCLIC GOLF,2.0,76.60,62.0,29.11,58.22
6,LC038,NaN,Lucas Classic Car Battery 536 201 031,2.0,77.54,59.0,31.79,63.58
7,895CXT,NaN,NUMAX PREMIUM CAR BATTERY,1.0,65.96,60.0,26.38,26.38
8,LV22MF,NaN,NUMAX DC LEISURE BATTERY NCC=C,1.0,117.23,57.0,50.41,50.41
9,LTX9BS,NaN,LUCAS 12V SEALED MOTORCYCLE,1.0,39.71,58.0,16.68,16.68


In [8]:
content.rename(columns={
    0: 'Product Code',
    1: 'Customer Code',
    2: 'Description',
    3: 'Qty',
    4: 'Price',
    5: 'Disc%',
    6: 'Nett Price',
    7: 'Line Total'}, inplace=True)

display(content)

,Product Code,Customer Code,Description,Qty,Price,Disc%,Nett Price,Line Total
0,LSLA4-6,NaN,LUCAS 6V 4AH AGM STANDBY BATTERY,40.0,9.89,62.0,3.76,150.40
1,LSLA7-12,NaN,LUCAS 12V 7AH AGM STANDBY,14.0,8.71,NaN,8.71,121.94
2,LSLA4.5-12,NaN,LUCAS 12V 4.5AH AGM STANDBY,16.0,19.57,62.0,7.44,119.04
3,LSLA5-12,NaN,LUCAS 12V 5AH AGM STANDBY,10.0,24.18,62.0,9.19,91.90
4,EK131,NaN,EXIDE Auxiliary Battery - EK131,2.0,69.54,54.0,31.99,63.98
5,LSLC22-12G,NaN,LUCAS 12V 22AH AGM CYCLIC GOLF,2.0,76.60,62.0,29.11,58.22
6,LC038,NaN,Lucas Classic Car Battery 536 201 031,2.0,77.54,59.0,31.79,63.58
7,895CXT,NaN,NUMAX PREMIUM CAR BATTERY,1.0,65.96,60.0,26.38,26.38
8,LV22MF,NaN,NUMAX DC LEISURE BATTERY NCC=C,1.0,117.23,57.0,50.41,50.41
9,LTX9BS,NaN,LUCAS 12V SEALED MOTORCYCLE,1.0,39.71,58.0,16.68,16.68


In [9]:
dict_content = content.to_dict(orient='records')
dict_content


line_items=[]
for item in dict_content:
    # print(item)
    
    partNum = item['Product Code']
    desc = item['Description']
    quantity = item['Qty']
    netTotal = item['Nett Price']

    print(partNum)
    
    line_item = {"line_type": "inventory",
                        "sku": partNum,
                        "name": desc,
                        "quantity": int(quantity),
                        "net_total": float(netTotal),
                        "tax_type": "INPUT2"}
    
    line_items.append(line_item)
    
print(line_items)

LSLA4-6
LSLA7-12
LSLA4.5-12
LSLA5-12
EK131
LSLC22-12G
LC038
895CXT
LV22MF
LTX9BS
EB16CL-B
LSLA3.2-12
EA612
YB12ALA2
LF027
EB705
E60-N24L-A
40-256
[{'line_type': 'inventory', 'sku': 'LSLA4-6', 'name': 'LUCAS 6V 4AH AGM STANDBY BATTERY', 'quantity': 40, 'net_total': 3.76, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'LSLA7-12', 'name': 'LUCAS 12V 7AH AGM STANDBY', 'quantity': 14, 'net_total': 8.71, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'LSLA4.5-12', 'name': 'LUCAS 12V 4.5AH AGM STANDBY', 'quantity': 16, 'net_total': 7.44, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'LSLA5-12', 'name': 'LUCAS 12V 5AH AGM STANDBY', 'quantity': 10, 'net_total': 9.19, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'EK131', 'name': 'EXIDE Auxiliary Battery - EK131', 'quantity': 2, 'net_total': 31.99, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'LSLC22-12G', 'name': 'LUCAS 12V 22AH AGM CYCLIC GOLF', 'quantity': 2, 'net_total': 29.11, 'tax_type'

In [10]:
table3 = read_pdf(input_file,
            pages=num_pages,
            silent=True,
            guess=False,
            area=(560,408,640,565),
            columns=[505,565],
            pandas_options={'header': None},
            encoding='windows-1254')

total_content=table3[0]

#total_content = total_content.dropna(subset=[0,1])  # Remove rows with NaN in column 0 & 1
#total_content = total_content.dropna(subset=[0]).reset_index(drop=True)
display(total_content)

row_index = total_content.index[total_content[0] == "Invoice Total GBP"].tolist()[0]

final_total = float(str(total_content[1][row_index]).replace(',',''))
display(final_total)

,0,1
0,Nett Total,"1,366.70"
1,VAT,273.36
2,Invoice Total GBP,"1,640.06"


1640.06

In [11]:
payload = {}
keys = ["Source File",
        "Type",
        "Name",
        "Date",
        "Reference No.",
        "Order No.",
        "Transfer No.",
        "Document No.",
        "Line Items",
        "Total"]

values = [file_name,
        invoice_type,
        name,
        date,
        docnum,
        ordernum,
        transfernum,
        None,
        line_items,
        final_total]

for i, key in enumerate(keys):
    payload[key] = values[i]

payload

{'Source File': 'C:\\Users\\admin\\Documents\\22.05.2023 £1,542.83 Ecobat Battery UK Ltd.pdf',
 'Type': 'Products',
 'Name': 'Ecobat Battery',
 'Date': '2023-10-11 00:00:00',
 'Reference No.': 23245004,
 'Order No.': 'PO27294',
 'Transfer No.': None,
 'Document No.': None,
 'Line Items': [{'line_type': 'inventory',
   'sku': 'LSLA4-6',
   'name': 'LUCAS 6V 4AH AGM STANDBY BATTERY',
   'quantity': 40,
   'net_total': 3.76,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'LSLA7-12',
   'name': 'LUCAS 12V 7AH AGM STANDBY',
   'quantity': 14,
   'net_total': 8.71,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'LSLA4.5-12',
   'name': 'LUCAS 12V 4.5AH AGM STANDBY',
   'quantity': 16,
   'net_total': 7.44,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'LSLA5-12',
   'name': 'LUCAS 12V 5AH AGM STANDBY',
   'quantity': 10,
   'net_total': 9.19,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'EK131',
   'name': 'EXIDE Auxi